# 04 - Claims Prediction: Frequency & Severity

Unlike `Risk_Category`, both targets modeled here are genuine columns
from the raw export: `Claim Freq` (did the policyholder claim?) and
`Claim Amount` (how much?). No engineered labels involved.


In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline


In [2]:
from src.data_prep import load_raw_data, clean_data
from src.features import engineer_features
from src.train_claims_models import train_claim_frequency_model, train_claim_amount_model
from src import config

df = engineer_features(clean_data(load_raw_data()))
print(f"Baseline claim rate: {df['Has_Claim'].mean()*100:.2f}%")


[data_prep] Dropped 1 duplicate policy rows.
Baseline claim rate: 27.54%


## Does anything predict who claims?

In [3]:
for col in ['Car_Use', 'Age_Group', 'Kids_Driving', 'Coverage_Zone']:
    print(df.groupby(col, observed=True)['Has_Claim'].mean().round(3), '\n')


Car_Use
Commercial    0.269
Private       0.277
Name: Has_Claim, dtype: float64 

Age_Group
18-25    0.289
26-35    0.273
36-45    0.271
46-55    0.279
56-65    0.270
65+      0.274
Name: Has_Claim, dtype: float64 

Kids_Driving
0    0.276
1    0.271
2    0.273
3    0.288
Name: Has_Claim, dtype: float64 

Coverage_Zone
Highly Rural    0.271
Highly Urban    0.279
Rural           0.272
Suburban        0.275
Urban           0.280
Name: Has_Claim, dtype: float64 



Claim rate barely moves across any of these, staying around 27-28%
regardless of usage type, age band, household composition, or geography.
That's the headline finding of this notebook, reported as-is rather than
hunting for a feature combination that looks more predictive than it
really is.


## Claim frequency classifier vs. a naive baseline

In [4]:
freq_results = train_claim_frequency_model(df)
import pandas as pd
pd.DataFrame(freq_results).T


[claim-freq | Baseline (majority class)] accuracy=72.46%  roc_auc=50.0


[claim-freq | Logistic Regression] accuracy=72.46%  roc_auc=50.18


[claim-freq | Random Forest] accuracy=72.46%  roc_auc=50.27


[figure saved] /sessions/quirky-funny-cori/mnt/outputs/UnderSure-AI/reports/figures/14_claim_frequency_comparison.png


,accuracy,f1,roc_auc
Baseline (majority class),72.46,0.0,50.00
Logistic Regression,72.46,0.0,50.18
Random Forest,72.46,0.0,50.27


## Claim amount regression vs. a naive baseline

In [5]:
amount_results = train_claim_amount_model(df)
pd.DataFrame(amount_results).T


[claim-amount | Baseline (mean amount)] MAE=$2464.05  R2=-0.0001
[claim-amount | Linear Regression] MAE=$2461.4  R2=-0.0003


[claim-amount | Random Forest] MAE=$2459.85  R2=0.0012


[figure saved] /sessions/quirky-funny-cori/mnt/outputs/UnderSure-AI/reports/figures/15_claim_amount_comparison.png


,mae,r2
Baseline (mean amount),2464.05,-0.0001
Linear Regression,2461.40,-0.0003
Random Forest,2459.85,0.0012


## Interpretation

Both models come in essentially tied with their naive baselines (majority
class / mean prediction). Rather than read this as a disappointing
result, it's a useful one: idiosyncratic claim risk (accidents, weather,
plain bad luck) dominates over anything derivable from demographic and
vehicle attributes alone. That's also a plausible reason real insurers
invest in telematics and claims-history data rather than leaning on
demographics.
